# Hospital Dermatology Vision Ensemble (224x224) — Kaggle GPU Training (100% From Scratch)

Trains a **3-seed `PyTorchDermResNet`** (`pretrained_weights_used = False`, CUDA FP16 mixed precision `AMP`, `body_site_dropout=0.40`) on NVIDIA GPU using the locked patient/lesion-grouped v3 splits (`configs/vision/val_v3.txt`, `configs/vision/locked_test_v3.txt`), fits `val_v3`-only per-modality calibration temperatures, and audits biopsy-confirmed (`histopathology`) AUROC.

### Required Kaggle Secrets (Add-ons -> Secrets)
- `DERM_CLOUDFRONT_USER`
- `DERM_CLOUDFRONT_PASS`

> **Privacy rule:** Public `derm-corpus` data only. Never upload real hospital patient photos to Kaggle.

In [ ]:
import torch, subprocess
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# 1. Clone repo & install lightweight project deps (keeping Kaggle preinstalled CUDA PyTorch)
import os, subprocess
REPO_URL = "https://github.com/Mtg221/medical_ai.git"
REPO_DIR = "/kaggle/working/medical_ai"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "httpx", "pyyaml", "pydantic", "pillow", "scikit-learn", "fastapi", "uvicorn", "argon2-cffi"], check=True)
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print("Ready in", os.getcwd())

In [ ]:
# 2. Load CloudFront credentials securely from Kaggle Secrets & sync metadata + balanced thumbnails
import os, subprocess

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["DERM_CLOUDFRONT_USER"] = secrets.get_secret("DERM_CLOUDFRONT_USER")
    os.environ["DERM_CLOUDFRONT_PASS"] = secrets.get_secret("DERM_CLOUDFRONT_PASS")
    print("Loaded DERM_CLOUDFRONT credentials from Kaggle Secrets (redacted).")
except Exception as exc:
    print("Warning: Kaggle Secrets unavailable — ensure DERM_CLOUDFRONT_USER/PASS are set in env:", type(exc).__name__)

subprocess.run(["medextract", "derm", "sync", "--meta-only"], check=True)
subprocess.run(["medextract", "derm", "sync", "--plan", "docs/data/DERM_SAMPLING_PLAN.md"], check=True)

In [ ]:
# 3. Train 3-seed 224x224 PyTorchDermResNet on CUDA GPU + val_v3 per-modality calibration + histopathology audit
import json, subprocess
from pathlib import Path

subprocess.run([
    "python", "scripts/train_vision_gpu.py",
    "--image-size", "224",
    "--seeds", "1,2,3",
    "--epochs", "25",
    "--batch-size", "64",
    "--lr", "0.001",
    "--patience", "6",
    "--body-site-dropout", "0.40",
    "--max-train-samples", "40000",
    "--max-samples-per-epoch", "24000",
    "--device", "auto",
    "--out-models-dir", "models",
    "--out-report", "docs/results/vision/vision_gpu_benchmark.json",
], check=True)

report = json.loads(Path("docs/results/vision/vision_gpu_benchmark.json").read_text(encoding="utf-8"))
print("Val-fitted modality temps:", report["val_fitted_modality_temperatures"])
print("Overall calibrated:", report["overall_per_modality_calibrated"])
print("By modality calibrated:", report["by_modality_calibrated"])
print("Histopathology-confirmed dermoscopic:", report["histopathology_confirmed_dermoscopic"])

In [ ]:
# 4. Package trained CPU-loadable Vision checkpoints + calibration file for download
import zipfile
from pathlib import Path

zip_path = Path("/kaggle/working/hospital_vision_models_gpu.zip")
targets = [
    Path("models/derm_ensemble_calibration.json"),
    Path("docs/results/vision/vision_gpu_benchmark.json"),
    *sorted(Path("models/derm_cnn_bsdrop40_seed1").glob("*")),
    *sorted(Path("models/derm_cnn_seed2").glob("*")),
    *sorted(Path("models/derm_cnn_seed3").glob("*")),
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in targets:
        if p.exists() and p.is_file():
            zf.write(p, arcname=str(p))
            print("Added:", p)
print(f"Created {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB) — download from Kaggle Output tab and unzip into repo root.")